Develop a scheme to parallelize the Elastic Net Regression algorithm as much as possible. 
Now, execute this scheme in CUDA if possible. Explain how hardware blocks could be designed for this regression inference implementation.

In [2]:
# Create the CUDA source file
cuda_code = '''#include <stdio.h>

// CUDA kernel - executes on GPU
__global__ void helloFromGPU() {
    printf("Hello from GPU (block %d, thread %d)\\n", blockIdx.x, threadIdx.x);
}

int main() {
    printf("Hello from CPU\\n");
    
    // Launch kernel with 1 block and 10 threads per block
    helloFromGPU<<<1, 10>>>();
    
    // Wait for GPU to finish
    cudaDeviceSynchronize();
    
    printf("Goodbye from CPU\\n");
    
    return 0;
}
'''

# Write to file
with open('hello_world.cu', 'w') as f:
    f.write(cuda_code)

print("CUDA source file created: hello_world.cu")

CUDA source file created: hello_world.cu


In [3]:
# Compile the CUDA program
import subprocess

result = subprocess.run(['nvcc', '-o', 'hello_world', 'hello_world.cu'], 
                       capture_output=True, text=True)

if result.returncode == 0:
    print("✓ Compilation successful!")
else:
    print("✗ Compilation failed:")
    print(result.stderr)

✓ Compilation successful!


In [4]:
# Run the compiled CUDA program
result = subprocess.run(['./hello_world'], capture_output=True, text=True)

print("Program Output:")
print("=" * 40)
print(result.stdout)
if result.stderr:
    print("Errors:")
    print(result.stderr)
print("=" * 40)

Program Output:
Hello from CPU
Hello from GPU (block 0, thread 0)
Hello from GPU (block 0, thread 1)
Hello from GPU (block 0, thread 2)
Hello from GPU (block 0, thread 3)
Hello from GPU (block 0, thread 4)
Hello from GPU (block 0, thread 5)
Hello from GPU (block 0, thread 6)
Hello from GPU (block 0, thread 7)
Hello from GPU (block 0, thread 8)
Hello from GPU (block 0, thread 9)
Goodbye from CPU



## Code Explanation

### CUDA Kernel `__global__ void helloFromGPU()`
- `__global__`: Keyword that marks this as a CUDA kernel (runs on GPU)
- `blockIdx.x`: The block index (which block this thread belongs to)
- `threadIdx.x`: The thread index within a block

### Kernel Launch `helloFromGPU<<<1, 10>>>()`
- `<<<1, 10>>>`: Configuration (blocks, threads per block)
  - 1 block
  - 10 threads per block
  - Total: 10 threads executing in parallel on the GPU

### `cudaDeviceSynchronize()`
- Blocks CPU execution until GPU finishes all work
- Essential for correct ordering of output

### Expected Output
Each thread prints its block and thread ID, demonstrating GPU parallelism!

In [2]:
!nvidia-smi

Fri Feb 20 07:36:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             54W /  400W |     448MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [13]:
import subprocess

# Create and compile CUDA code
vector_add_code = '''#include <iostream>
__global__ void helloCUDA() {
   printf("Hello from GPU thread %d\\n", threadIdx.x);
}
int main() {
   helloCUDA<<<1, 5>>>();
   cudaDeviceSynchronize();
   return 0;
}
'''

with open('hello_cuda.cu', 'w') as f:
    f.write(vector_add_code)

result = subprocess.run(['nvcc', '-o', 'hello_cuda', 'hello_cuda.cu'], 
                       capture_output=True, text=True)

if result.returncode == 0:
    result = subprocess.run(['./hello_cuda'], capture_output=True, text=True)
    print(result.stdout)
else:
    print("Compilation failed:", result.stderr)

Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3
Hello from GPU thread 4



In [20]:
# Find max element using CUDA
find_max_code = '''#include <iostream>
using namespace std;
__global__ void findMax(int* a, int* b, int n) {
   int idx = threadIdx.x + blockIdx.x * blockDim.x;
   if (idx < n) atomicMax(b, a[idx]);
}
int main() {
   int n = 10;
   int h_a[10] = {1, 5, 3, 9, 2, 8, 20, 6, 4, 7};
   int h_max = 0, *d_a, *d_max;
   cudaMalloc(&d_a, n * sizeof(int));
   cudaMalloc(&d_max, sizeof(int));
   cudaMemcpy(d_a, h_a, n * sizeof(int), cudaMemcpyHostToDevice);
   cudaMemcpy(d_max, &h_max, sizeof(int), cudaMemcpyHostToDevice);
   findMax<<<1, n>>>(d_a, d_max, n);
   cudaMemcpy(&h_max, d_max, sizeof(int), cudaMemcpyDeviceToHost);
   cout << "Max value: " << h_max << endl;
   cudaFree(d_a); cudaFree(d_max);
   return 0;
}
'''

with open('find_max.cu', 'w') as f:
    f.write(find_max_code)

result = subprocess.run(['nvcc', '-o', 'find_max', 'find_max.cu'], 
                       capture_output=True, text=True)

if result.returncode == 0:
    result = subprocess.run(['./find_max'], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("Errors:", result.stderr)
else:
    print("Compilation failed:", result.stderr)

Max value: 20

